# Kepler-8 transit analysis

Detecting the transit of Kepler-8 b in Kepler lightcurve data using Python, lightkurve and BLS (Box Least Squares).

Steps: download lightcurve, bin, flatten, BLS period search, fold, plot.

In [ ]:
!pip install lightkurve -q
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

lc = lk.search_lightcurve("Kepler-8", mission="Kepler")[0].download().remove_nans().normalize()
lc.plot()
plt.savefig("1_raw_lightcurve.png", dpi=150)
plt.show()

lc30 = lc.bin(time_bin_size=0.0208)
flat = lc30.flatten(window_length=401).remove_outliers(sigma_upper=4, sigma_lower=float("inf"))
flat.plot()
plt.savefig("2_flattened.png", dpi=150)
plt.show()

period = flat.to_periodogram("bls", period=np.arange(1, 10, 0.001), duration=[0.1, 0.15, 0.2, 0.25])
print("Period:", period.period_at_max_power)
print("Depth:", period.depth_at_max_power)
print("Duration:", period.duration_at_max_power)

folded = flat.fold(period=period.period_at_max_power, epoch_time=period.transit_time_at_max_power)
ph = np.ma.filled(folded.time.value, np.nan).astype(float)
fl = np.ma.filled(folded.flux.value, np.nan).astype(float)
ok = np.isfinite(ph) & np.isfinite(fl)
ph, fl = ph[ok], fl[ok]
idx = np.digitize(ph, np.arange(ph.min(), ph.max(), 0.02))
bx = [ph[idx == i].mean() for i in np.unique(idx)]
by = [fl[idx == i].mean() for i in np.unique(idx)]

plt.scatter(ph, fl, s=1, alpha=0.3, color="gray")
plt.plot(bx, by, "r.-")
plt.xlim(-0.5, 0.5)
plt.xlabel("Phase (days)")
plt.ylabel("Normalized flux")
plt.savefig("fold_plot.png", dpi=150)
plt.show()

res = {"period_days": [period.period_at_max_power.value], "depth": [period.depth_at_max_power.value], "duration_days": [period.duration_at_max_power.value]}
pd.DataFrame(res).to_csv("results.csv", index=False)

## Result

BLS period: 3.524 days, depth: about 0.84%, duration: about 0.1 days (2.4 hours). The period matches the known planet Kepler-8 b.